# Magic Leap 2 Navigation - Thesis Process Documentation

This notebook documents the complete process involved in creating the Magic Leap 2 (ML2) Navigation Library.

## Background

In recent years, several computer-assisted surgery at the Robarts Research Institute have relied on the Microsoft HoloLens 2 (HL2), an augmented reality (AR) head-mounted display (HMD) for real-time surgical tool-tracking and visualization. These projects use a Unity-based plugin ([here](https://github.com/andreaskeller96/HoloLens2-IRTracking-Sample)) to estimate the pose (position and orientation) of surgical tools by detecting the retro-reflective markers attached to them. 

Following the discontinuation of the HoloLens 2 in 2024, researchers in the lab have begun exploring migration to the Magic Leap 2 (ML2) as an alternative AR platform. However, the lack of an equivalent tool-tracking Unity plugin for the ML2 currently prevents existing systems from being ported to the new device. As a result, developing a compatible tracking framework for the ML2 has become an important prerequisite for continuing these computer-assisted surgery projects.

## System Overview

![Video of live tracking](images/retro-reflective.png)

The project is split into 4 sections:
1. ML2 Setup – For Unity and C projects

2. Marker Detection
   1. Detect markers using IR intensity and blob detection
   2. Track points in camera space (relative to the camera)

3. Tool Definition
   1. Convert points into world space
   2. Create a unique signature for each tool by computing pairwise marker distances

4. Tool Tracking
   1. Apply Kalman filtering each frame for prediction
   2. Identify tools in real time using a graph-based search

## Terms

Figure 1: Retro-reflective Markers
![Retro Reflective Marker](images/retro-reflective.png)

- Retro-reflective Markers (RRM): markers that reflect infrared light directly back to the camera sensor. This makes them appear as bright blobs in infrared images, allowing them to be easily detected using computer vision algorithms.
- ToF Camera:
- 6 Degrees of Freedom (6DoF):
- Kalman Filters:





# 1. ML2 Setup

Since the ML2 is very reliant on its dependencies, it is important to have the correct versions and paths as follows.

### Reference for versions:
- Unity Editor: 2022.3
- Minimum API Level for Unity: 29
- Android SDK 10 (Q) API Level 29
- Android NDK, v 25.0.8775105
- CMake v 3.22.1
- MLSDK 12 (on ML Hub 3)
- Clang (need to check what that issue was for the sample) TODO
- Java 11 (what version is needed) TODO
- Magic Leap OS 12.1

### Path examples:
- export MLSDK="$HOME/MagicLeap/mlsdk/v1.12.0"
- export MAGICLEAP_APP_FRAMEWORK_ROOT="$MLSDK/samples/c_api/app_framework" - NOTE: specific to C project
- export ANDROID_HOME="$HOME/Library/Android/sdk"
- export JAVA_HOME=/Library/Java/JavaVirtualMachines/openjdk-11.jdk/Contents/Home
- VULKAN_SDK, VULKAN_PATH -> TODO: I need to do this

### Setup Steps:

1. [Install the tools](https://developer-docs.magicleap.cloud/docs/guides/unity-openxr/getting-started/install-the-tools/)
   1. Install ML Hub 3, Unity Hub
   2. Download the Unity Example projects (2.6)
   3. Install the Unity Editor 2022.3 (NOTE: later versions may not work) and relevant modules (Android Build Support, Android SDK & NDK Tools, OpenJDK)
2. [Environment Setup for Android](https://developer-docs.magicleap.cloud/docs/guides/openxr/getting-started/openxr-envsetup/)
3. [Follow the ML2 Tutorial on Unity, specifically 2 and 3](https://learn.unity.com/course/magic-leap-2-development/tutorial/get-started-creating-for-the-magic-leap-2?version=2022.2). 
   1. NOTE: Download MLSDK version 1.12 from ML Hub 3.
   2. NOTE: Ignore the setup for the ML2 application simulator


Background:

- I am working on transferring a navigation library from the Hololens 2 to the Magic leap 2. 

Setup Background:

- I am following the setup instructions for the magic leap 2's openXR unity documentation
- This has you download the magic leap 2's package, which includes the OpenXR plugin (1.13)

release documentation:
https://developer-docs.magicleap.cloud/docs/releases/releases-overview/index.html


https://learn.unity.com/course/magic-leap-2-development/tutorial/get-started-creating-for-the-magic-leap-2?version=2022.2

OS version: 1.12.1

Unity version: 2022.3.32f1
Magic leap SDK: 2.6.0 pre.R15
    - OpenXR plugin (1.13) downloaded as a dependency
Magic leap XR plugin: 7.0.0
XR interaction Toolkit: 2.5.4 (not necessary per say)

- Platform: Android

MAgicLeap Permissions: API Level 31
    - every permission is on

Make sure you're building the correct scene in build settings
- to see if you're genuinely running a different version: update product name 
    - build settings -> player settings -> product name

- try using ML Rig and OpenXR input from samples under magic leap SDK as a test
    - Validate that there's a unity logo
    - Validate that the object is anchored in place

# Building an Native Android plugin

## Key information

The conversion of this project focused heavily on 3 types of changes 1) build type (windows to android) 2) API (HL2 to ML2 SDK) 3) headset specific (differences in sensors)


## Architecture

I chose to build my application as a native plug-in for unity. This plug-in would work access the headset directly through ML2's SDK

Unity C# under IL2CPP is already native

Native plugins help when:

big buffers

CV / sensor pipelines

SIMD

no GC

vendor SDKs

Performance comes from:

memory layout

fewer transitions

compile flags

threading

vectorization

# 2. Marker Detection 

Utilizes OpenCV + ML2 instrisics 

## Section A - Preprocessing the intensity image

Goal: To make the intensity 

1. subtracts the ambient image buffer (ambient intensity of each pixel in the image) and clamps the value to 0 so subtraction can't create negatives)

Why: retro-reflective markers should remain bright while ambient/IR background can be reduced



In [ ]:
cv::Mat intensity_image;
if (config.use_ambient_subtraction && ambient_intensity_buffer != nullptr) {
    cv::Mat ambient(height, width, CV_32FC1, ambient_intensity_buffer);
    intensity_image = raw_intensity - ambient;
    cv::threshold(intensity_image, intensity_image, 0, 0, cv::THRESH_TOZERO);
} else {
    intensity_image = raw_intensity;
}

## Section B - Creating bright segments and countering noise

